### https://www.kaggle.com/competitions/drawing-with-llms

In [1]:
import kagglehub
import pandas as pd
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import sys
sys.path.append('./utils')

### Random seed for reproducibility

In [2]:
import torch
import random
import numpy as np
#import multiprocessing as mp
#mp.set_start_method("spawn", force=True)
# Now set your seed
seed = 123
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [3]:
import concurrent
import io
import logging
import re
import re2
import cairosvg
import kagglehub
from lxml import etree
from vllm import LLM, SamplingParams
import gc
import svg_constraints 
from svg_processor import SVGSanitizer, SVGProcessor

class Model:
    
    def __init__(self):

        self.model_path="./lora/Llama_32_3B_Instruct_lora_fp16_r256_s10000_i2000_msl2048"
        self.model = LLM(
            model=self.model_path,
            max_model_len=1024,
            #quantization="AWQ",
            gpu_memory_utilization=0.85,
            dtype="half",
            seed=123,
            disable_log_stats=True
        )
       
        self.default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        self.constraints = svg_constraints.SVGConstraints()
        self.sanitizer = SVGSanitizer(self.constraints, self.default_svg)

    def close_model(self):
        del self.model 
        gc.collect()     


    def _format_prompt(self, description: str) -> str:
        return  f"""Below is an instruction that describes a task, paired with an input that provides further context. 
                Write a response that appropriately completes the request.
                
                ### Instruction:
                Generate a SVG code for the given input:
                
                ### Input:
                {description}
                
                ### Response:
                """
    
    def get_response(self, descriptions):
        
        formatted_input = [self._format_prompt(desc) for desc in descriptions]
        sampling_params = SamplingParams(temperature=0.7, top_p=0.95, max_tokens=1024,n=1)
        outputs = self.model.generate(formatted_input, sampling_params)
        
        #suitable for batch inputs as well
        output_list=[]
        for output in outputs:
            prompt = output.prompt
            generated_text = output.outputs[0].text
            output_list.append(generated_text.strip())
        return output_list
    
    def predict(self, descriptions: list[str], max_new_tokens=1024) -> list[str]:
        output_decoded_list = self.get_response(descriptions)
        final_svg_code_list = []
    
        for description, output in zip(descriptions, output_decoded_list):
            base_svg = SVGProcessor.clean_and_extract_svgs(output, self.default_svg)
            #clean_svg = self.sanitizer.enforce_constraints(base_svg)
            final_svg = SVGProcessor.svg_conversion_check(description, base_svg, self.default_svg)
            final_svg_code_list.append(final_svg)
    
        return final_svg_code_list


INFO 05-01 22:20:31 [__init__.py:239] Automatically detected platform cuda.


In [4]:
model=Model()

WARNING 05-01 22:20:32 [config.py:2614] Casting torch.bfloat16 to torch.float16.
INFO 05-01 22:20:36 [config.py:585] This model supports multiple tasks: {'reward', 'classify', 'score', 'embed', 'generate'}. Defaulting to 'generate'.
INFO 05-01 22:20:36 [config.py:1697] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-01 22:20:37 [core.py:54] Initializing a V1 LLM engine (v0.8.2) with config: model='./lora/Llama_32_3B_Instruct_lora_fp16_r256_s10000_i2000_msl2048', speculative_config=None, tokenizer='./lora/Llama_32_3B_Instruct_lora_fp16_r256_s10000_i2000_msl2048', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=1024, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 05-01 22:20:41 [loader.py:447] Loading weights took 2.35 seconds
INFO 05-01 22:20:41 [gpu_model_runner.py:1186] Model loading took 6.0160 GB and 2.509226 seconds
INFO 05-01 22:20:47 [backends.py:415] Using cache directory: /home/vino/.cache/vllm/torch_compile_cache/b26470748b/rank_0_0 for vLLM's torch.compile
INFO 05-01 22:20:47 [backends.py:425] Dynamo bytecode transform time: 5.95 s
INFO 05-01 22:20:47 [backends.py:115] Directly load the compiled graph for shape None from the cache
INFO 05-01 22:20:52 [monitor.py:33] torch.compile takes 5.95 s in total
INFO 05-01 22:20:53 [kv_cache_utils.py:566] GPU KV cache size: 17,392 tokens
INFO 05-01 22:20:53 [kv_cache_utils.py:569] Maximum concurrency for 1,024 tokens per request: 16.98x
INFO 05-01 22:21:08 [gpu_model_runner.py:1534] Graph capturing finished in 15 secs, took 0.44 GiB
INFO 05-01 22:21:08 [core.py:151] init engine (profile, create kv cache, warmup model) took 26.76 seconds


In [5]:
#model.predict(['who are you?'])

In [6]:
import sys
sys.path.append(r'/home/vino/ML_Projects/Drawing_with_LLMs/drawing-with-llms')
import pandas as pd

df1=pd.read_csv(r'./drawing-with-llms/test_filtered_1_batch_vqa_gpt4.csv',header=[0])
df2=pd.read_csv(r'./drawing-with-llms/test_filtered_2_batch_vqa_gemini_2o_kaggle.csv',header=[0])
#df3=pd.read_csv(r'./drawing-with-llms/gemini_25_pro_validation/train_filtered_1_batch_gpt4.csv',header=[0])
#print(df3.shape)
df2=df2.drop_duplicates(['description'])
df=pd.concat([df1[['description']],df2['description']],axis=0)
df=df.drop_duplicates(['description'])

print(df.shape)
df.head(2)

(170, 1)


,description
0,"'Vibrant autumn forest',"
1,"'Morning dew on grass',"


In [7]:
# from tqdm import tqdm
# tqdm.pandas()
# df['svg_3'] = df['description'].progress_apply(lambda x: model.predict(x))

In [8]:
from tqdm import tqdm
description_list = [s.strip(" ',") for s in df['description'].to_list()]
batch_size = 15
results = []

for i in tqdm(range(0, len(description_list), batch_size), desc="Batch prediction"):
    batch = description_list[i:i + batch_size]
    batch_result = model.predict(batch)  # Ensure this handles a list of inputs
    results.extend(batch_result)


Batch prediction:   0%|                                  | 0/12 [00:00<?, ?it/s]
cessed prompts:   0%| | 0/15 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, 
cessed prompts:   7%| | 1/15 [00:02<00:38,  2.74s/it, est. speed input: 21.89
cessed prompts:  13%|▏| 2/15 [00:04<00:27,  2.09s/it, est. speed input: 27.45
cessed prompts:  20%|▏| 3/15 [00:04<00:15,  1.29s/it, est. speed input: 38.44
cessed prompts:  33%|▎| 5/15 [00:05<00:08,  1.24it/s, est. speed input: 54.28
cessed prompts:  47%|▍| 7/15 [00:07<00:07,  1.06it/s, est. speed input: 54.54
cessed prompts:  53%|▌| 8/15 [00:07<00:05,  1.31it/s, est. speed input: 61.05
cessed prompts:  67%|▋| 10/15 [00:08<00:03,  1.63it/s, est. speed input: 69.3
cessed prompts:  73%|▋| 11/15 [00:10<00:03,  1.11it/s, est. speed input: 62.7
cessed prompts:  80%|▊| 12/15 [00:11<00:02,  1.17it/s, est. speed input: 64.3
Processed prompts: 100%|█| 15/15 [00:13<00:00,  1.09it/s, est. speed input: 66.8
Batch prediction:   8%|██▏                       | 1/12 [0

Failed to convert Quiet forest pathway due to not well-formed (invalid token): line 47, column 29, Returning default SVG.



cessed prompts:   0%| | 0/15 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, 
cessed prompts:   7%| | 1/15 [00:03<00:50,  3.61s/it, est. speed input: 17.44
cessed prompts:  13%|▏| 2/15 [00:04<00:28,  2.20s/it, est. speed input: 25.74
cessed prompts:  20%|▏| 3/15 [00:05<00:15,  1.30s/it, est. speed input: 37.03
cessed prompts:  27%|▎| 4/15 [00:05<00:09,  1.13it/s, est. speed input: 47.17
cessed prompts:  33%|▎| 5/15 [00:05<00:06,  1.49it/s, est. speed input: 55.82
cessed prompts:  40%|▍| 6/15 [00:05<00:04,  2.03it/s, est. speed input: 65.58
cessed prompts:  47%|▍| 7/15 [00:06<00:04,  1.75it/s, est. speed input: 67.73
cessed prompts:  53%|▌| 8/15 [00:08<00:06,  1.04it/s, est. speed input: 60.49
cessed prompts:  60%|▌| 9/15 [00:08<00:04,  1.31it/s, est. speed input: 65.50
cessed prompts:  67%|▋| 10/15 [00:09<00:04,  1.22it/s, est. speed input: 65.4
cessed prompts:  73%|▋| 11/15 [00:10<00:03,  1.19it/s, est. speed input: 65.8
cessed prompts:  80%|▊| 12/15 [00:12<00:03,  1.33s/it, est. spe

Failed to convert Windy wheat fields due to not well-formed (invalid token): line 41, column 67, Returning default SVG.
Failed to convert Autumn forest with falling leaves due to not well-formed (invalid token): line 49, column 60, Returning default SVG.



cessed prompts:   0%| | 0/15 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, 
cessed prompts:   7%| | 1/15 [00:03<00:53,  3.83s/it, est. speed input: 16.71
cessed prompts:  13%|▏| 2/15 [00:05<00:33,  2.54s/it, est. speed input: 23.40
cessed prompts:  20%|▏| 3/15 [00:05<00:18,  1.54s/it, est. speed input: 33.29
cessed prompts:  27%|▎| 4/15 [00:06<00:14,  1.29s/it, est. speed input: 38.49
cessed prompts:  40%|▍| 6/15 [00:07<00:06,  1.38it/s, est. speed input: 54.01
cessed prompts:  53%|▌| 8/15 [00:07<00:03,  2.02it/s, est. speed input: 67.87
cessed prompts:  60%|▌| 9/15 [00:07<00:02,  2.13it/s, est. speed input: 73.02
cessed prompts:  67%|▋| 10/15 [00:08<00:01,  2.64it/s, est. speed input: 80.0
cessed prompts:  73%|▋| 11/15 [00:08<00:01,  2.08it/s, est. speed input: 80.1
cessed prompts:  80%|▊| 12/15 [00:09<00:01,  2.25it/s, est. speed input: 84.2
cessed prompts:  87%|▊| 13/15 [00:11<00:02,  1.00s/it, est. speed input: 71.6
Processed prompts: 100%|█| 15/15 [00:13<00:00,  1.10it/s, est. 

...

Failed to convert a vibrant coral reef teeming with fish due to not well-formed (invalid token): line 58, column 42, Returning default SVG.
Failed to convert a clockwork castle under a blood-red sky due to not well-formed (invalid token): line 47, column 40, Returning default SVG.



cessed prompts:   0%| | 0/15 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, 
cessed prompts:   7%| | 1/15 [00:02<00:28,  2.01s/it, est. speed input: 32.91
cessed prompts:  13%|▏| 2/15 [00:04<00:32,  2.47s/it, est. speed input: 27.91
cessed prompts:  20%|▏| 3/15 [00:04<00:16,  1.40s/it, est. speed input: 39.98
cessed prompts:  27%|▎| 4/15 [00:05<00:10,  1.06it/s, est. speed input: 50.56
cessed prompts:  33%|▎| 5/15 [00:05<00:08,  1.17it/s, est. speed input: 55.76
cessed prompts:  40%|▍| 6/15 [00:06<00:05,  1.58it/s, est. speed input: 64.63
cessed prompts:  47%|▍| 7/15 [00:09<00:11,  1.40s/it, est. speed input: 50.57
cessed prompts:  53%|▌| 8/15 [00:09<00:07,  1.06s/it, est. speed input: 55.91
cessed prompts:  60%|▌| 9/15 [00:09<00:04,  1.29it/s, est. speed input: 61.88
cessed prompts:  67%|▋| 10/15 [00:11<00:05,  1.02s/it, est. speed input: 59.2
cessed prompts:  73%|▋| 11/15 [00:11<00:03,  1.07it/s, est. speed input: 61.4
Processed prompts: 100%|█| 15/15 [00:13<00:00,  1.14it/s, est. 

Failed to convert a black and white photograph of a city street due to not well-formed (invalid token): line 34, column 10, Returning default SVG.
Failed to convert a digital illustration of a futuristic cityscape at night due to not well-formed (invalid token): line 41, column 47, Returning default SVG.
Failed to convert a rusty red pickup truck parked in a field due to not well-formed (invalid token): line 49, column 51, Returning default SVG.
Failed to convert a bowl of fresh blueberries due to mismatched tag: line 42, column 97, Returning default SVG.



cessed prompts:   0%| | 0/5 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, o
cessed prompts:  20%|▏| 1/5 [00:02<00:09,  2.39s/it, est. speed input: 28.48 
cessed prompts:  40%|▍| 2/5 [00:04<00:05,  1.97s/it, est. speed input: 32.68 
cessed prompts:  60%|▌| 3/5 [00:04<00:02,  1.12s/it, est. speed input: 46.72 
cessed prompts:  80%|▊| 4/5 [00:07<00:02,  2.09s/it, est. speed input: 33.54 
Processed prompts: 100%|█| 5/5 [00:09<00:00,  1.99s/it, est. speed input: 32.59 
Batch prediction: 100%|█████████████████████████| 12/12 [02:34<00:00, 12.88s/it]

Failed to convert a chartreuse spiral staircase due to could not convert string to float: 'A', Returning default SVG.


In [9]:
df['svg_3']=results

In [10]:
model.close_model()

In [11]:
from siglip_class import SVGMetricEvaluator
from aesthetic_evaluator import AestheticEvaluator

In [12]:
#SigLip Score
from tqdm import tqdm
tqdm.pandas()
evaluator = SVGMetricEvaluator()
df['svg_score_3'] = df.progress_apply(lambda row: evaluator.svg_metric(row['description'], row['svg_3']), axis=1)

100%|█████████████████████████████████████████| 170/170 [00:10<00:00, 16.18it/s]


In [13]:
#Aes Score
from tqdm import tqdm
tqdm.pandas()
aes_eval = AestheticEvaluator()
df['aes_score_3'] = df.progress_apply(lambda row: aes_eval.get_score(row['svg_3']), axis=1)

100%|█████████████████████████████████████████| 170/170 [00:18<00:00,  9.27it/s]


In [14]:
#combined score
df['combined_score_3'] = (df['svg_score_3']+df['svg_score_3']+df['aes_score_3'])/3

In [15]:
print('mean_svg_score:',df['svg_score_3'].mean(),'mean_aes_score:',df['aes_score_3'].mean(),'combined_score:',df['combined_score_3'].mean())

mean_svg_score: 0.3511502732334382 mean_aes_score: 0.4444852203481337 combined_score: 0.3822619222716701


In [16]:
default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
df_default_svg=df[df['svg_3']==default_svg]
print('default_svg_count:',df_default_svg.shape[0])
print('default_svg_score_mean:',df_default_svg['svg_score_3'].mean(),'default_aes_score_mean:',df_default_svg['aes_score_3'].mean(),\
     'combined_score:',df_default_svg['combined_score_3'].mean())

default_svg_count: 20
default_svg_score_mean: 6.305735580219851e-08 default_aes_score_mean: 0.4369946956634522 combined_score: 0.14566494059272123


In [17]:
df_non_default_svg=df[df['svg_3']!=default_svg]
print('non-default_svg_count:',df_non_default_svg.shape[0])
print('non-default_svg_score_mean:',df_non_default_svg['svg_score_3'].mean(),\
      'non-default_aes_score_mean:',df_non_default_svg['aes_score_3'].mean(),\
        'combined_score:',df_non_default_svg['combined_score_3'].mean())

non-default_svg_count: 150
non-default_svg_score_mean: 0.39797030125691585 non-default_aes_score_mean: 0.44548395697275794 combined_score: 0.4138081864955299
